In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
from transformers import RTDetrV2ForObjectDetection
print("OK!")

: 

In [2]:
from transformers import RTDetrV2ForObjectDetection, RTDetrImageProcessor
from PIL import Image
import torch

/home/talt_wireten_c/miniconda3/envs/canenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: /home/talt_wireten_c/miniconda3/envs/canenv/lib/python3.10/site-packages/torch/lib/libtorch_cpu.so: undefined symbol: iJIT_NotifyEvent

In [ ]:
device = "cuda:2"

In [ ]:
processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_v2_r50vd")
model = RTDetrV2ForObjectDetection.from_pretrained(
    "PekingU/rtdetr_v2_r50vd",
    torch_dtype=torch.float16
).to(device).eval()

print(f"Model on {device}")

In [ ]:
image = Image.open("/home/talt_wireten_c/road-segmentation/datasets/coco/images/train2017/000000581419.jpg")

# Detection
inputs = processor(images=image, return_tensors="pt").to(device=device, dtype=torch.float16)
with torch.no_grad():
    outputs = model(**inputs)

# Post-process
target_sizes = torch.tensor([image.size[::-1]], device=device)
results = processor.post_process_object_detection(
    outputs, target_sizes=target_sizes, threshold=0.5
)

# Sonuçları yazdır
for score, label, box in zip(results[0]["scores"], results[0]["labels"], results[0]["boxes"]):
    print(f"{model.config.id2label[label.item()]}: {score:.2f} @ {[round(b, 1) for b in box.tolist()]}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(image)

for score, label, box in zip(results[0]["scores"], results[0]["labels"], results[0]["boxes"]):
    x1, y1, x2, y2 = box.tolist()
    name = model.config.id2label[label.item()]
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor="red", facecolor="none")
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"{name}: {score:.2f}", color="white", fontsize=10,
            bbox=dict(facecolor="red", alpha=0.7))

ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
from segment_anything import sam_model_registry, SamPredictor

sam = sam_model_registry["vit_h"](checkpoint="/home/talt_wireten_c/road-segmentation/sam_vit_h_4b8939.pth")
sam = sam.to(device)
sam_predictor = SamPredictor(sam)
print("SAM yüklendi!")